# Creating the Amino Acid Buffers for Fusion Gene Analysis

Because domain boundaries have some leeway between proteins, we want to ensure each domain has some buffer when included in a fusion gene. We have found that if we are super strict (i.e. no buffer) then common fusion genes like BCR-ABL and FGFR3-TACC3 will have slight differences in their domain architecture than expected. So we need some buffers to be safe. We have manually a small subset of these because other databases with the domain architectures do not have easily accessed predictions.

In [15]:
import dansy
import pandas as pd
import numpy as np

In [30]:
ref = dansy.import_proteome_files(ref_file_dir='data/Current_Human_Proteome', ref_file_suffix='0715.csv')
domains = dansy.ngramUtilities.extract_unigrams(ref)

In [31]:
# Now go through all the domain architecture information to get an average domain length to determine the buffer size
dom_sizes = {}
for _,row in ref.iterrows():
    
    arch_info = row['Interpro Domains'].split(';')
    if arch_info != ['']:  
        for dom_info in arch_info:
            _,dom,start,end = dom_info.split(':')
            length = int(end)-int(start)
            if dom in dom_sizes:
                dom_sizes[dom].append(length)
            else:
                dom_sizes[dom] = [length]

In [32]:
# Let's double check that all the domains are present
set(domains).difference(dom_sizes.keys())

{''}

That's all good so now do the average for all of those and place them into a dataframe

In [33]:
mean_lengths = {k:np.mean(v) for k,v in dom_sizes.items()}
df=pd.DataFrame.from_dict(mean_lengths, orient='index').reset_index()
df.rename(columns={'index':'InterPro_ID', 0:'mean_length'},inplace=True)

In [34]:
min(mean_lengths.items(), key = lambda x:x[1])

('IPR001709', 8.380952380952381)

In [35]:
# Now create the buffer where it is either 2 or 5 depending on if it has a length >50 or not
df['aa_buffer'] = np.where(df['mean_length'] <= 50 ,2,5)

In [36]:
df.to_csv('domain_fusions_aa_buffers.csv', index=False)